# Основные метрики для каждого пайплайна

In [1]:
import sys
import os
import json

sys.path.append('./../')

In [2]:
import numpy as np
import pandas as pd
from dotenv import load_dotenv

from eval.self_repeatness import count_self_repeating_part
from eval.variarity import count_fields_variance
from eval.md_classes import MDField, DataStep

load_dotenv()

True

In [3]:
QUERIES_CNT = 16
RUNS = 8

## Проход по таблицам полей и оценка основных параметров

Оцениваем базу:
- Число полей, число токенов, время

In [4]:
def stats_counter(df: pd.DataFrame, q: int, r: int):
    fields_cnt = 0
    time = 0
    tokens_prompt = 0
    tokens_complete = 0
    tokens_total = 0

    df = df.loc[(df['query_id'] == q) & (df['run_id'] == r)]
    fields_cnt = len(df)
    time = df.iloc[0]['elapsed_time']

    if 'tokens' in df.columns: 
        tokens = df.iloc[0]['tokens']
        tokens = eval(tokens)
        tokens_prompt = tokens['prompt']
        tokens_complete = tokens['completion']
        tokens_total = tokens['total']
    else:
        tokens_prompt = df.iloc[0]['tokens_prompt']
        tokens_complete = df.iloc[0]['tokens_compl']
        tokens_total = df.iloc[0]['tokes_total']

    return {
        'fields_cnt': fields_cnt,
        'time': time,
        'tokens_prompt': tokens_prompt,
        'tokens_complete': tokens_complete,
        'tokens_total': tokens_total
    }


In [5]:
md_tables = {
    'm1': './../data/md_generations/m1_synth_bio_mds.csv',
    'm2': './../data/md_generations/m2_synth_bio_mds.csv',
    'm2_fixed': './../data/md_generations/m2_synth_bio_mds_fixed.csv',
    'm3': './../data/md_generations/m3_synth_bio_mds/m3_synth_bio_mds.csv',
    'm4': './../data/md_generations/m4_synth_bio_mds/m4_synth_bio_mds.csv',
}

overall_stats = []

for p_name, p_path in md_tables.items():
    df = pd.read_csv(p_path)

    fields = []
    tokens_total = []
    tokens_compl = []
    times = []

    for q_id in range(QUERIES_CNT):
        for r_id in range(RUNS):
            stats = stats_counter(df, q_id, r_id)
            fields.append(stats['fields_cnt'])
            tokens_total.append(stats['tokens_total'])
            tokens_compl.append(stats['tokens_complete'])
            times.append(stats['time'])
    
    overall_stats.append({
        'exp_name': p_name, 
        'fields_cnt_avg': np.array(fields).mean(), 
        'fields_cnt_std': np.array(fields).std(), 
        'total_tkns_avg': np.array(tokens_total).mean(), 
        'total_tkns_std': np.array(tokens_total).std(), 
        'compl_tkns_avg': np.array(tokens_compl).mean(), 
        'compl_tkns_std': np.array(tokens_compl).std(), 
        'time_avg': np.array(times).mean(), 
        'time_std': np.array(times).std()
    })

stats_df = pd.DataFrame(overall_stats)
stats_df.head(10)

,exp_name,fields_cnt_avg,fields_cnt_std,total_tkns_avg,total_tkns_std,compl_tkns_avg,compl_tkns_std,time_avg,time_std
0,m1,64.492188,14.997914,3026.656250,416.679423,2270.031250,413.745788,16.219037,2.916136
1,m2,56.843750,15.288597,9899.812500,1866.314675,8667.187500,1854.824115,58.318864,79.158930
2,m2_fixed,56.843750,15.288597,9899.812500,1866.314675,8667.187500,1854.824115,46.553239,11.551009
3,m3,143.820312,30.507794,162929.375000,87749.997933,11930.773438,3274.532004,96.592262,28.005361
4,m4,226.554688,86.928743,739081.210938,544790.800283,25900.601562,9734.406559,205.737294,73.951572


## Теперь метрики M2, M3, M4 - число шагов

In [6]:
def count_nodes_per_run_m2(df: pd.DataFrame, q_id: int, runs_cnt: int):
    nodes_cnt_per_run = []

    for r in range(runs_cnt):
        loc_df = df.loc[(df['query_id'] == q_id) & (df['run_id'] == r)]
        nodes_set = set()
        
        for r_id, row in loc_df.iterrows():
            row_nodes = row['dag_step']
            row_nodes = set(row_nodes)
            nodes_set = nodes_set.union(row_nodes)

        nodes_cnt_per_run.append(len(nodes_set))

    return nodes_cnt_per_run

def count_nodes_per_run_m34(df: pd.DataFrame, q_id: int, runs_cnt: int):
    nodes_cnt_per_run = []

    for r in range(runs_cnt):
        loc_df = df.loc[(df['query_id'] == q_id) & (df['run_id'] == r)]
        nodes_set = set()
        
        for r_id, row in loc_df.iterrows():
            node_name = row['node_name']
            nodes_set.add(node_name)

        nodes_cnt_per_run.append(len(nodes_set))

    return nodes_cnt_per_run

In [7]:
md_tables = {
    'm2': './../data/md_generations/m2_synth_bio_mds.csv',
    'm3': './../data/md_generations/m3_synth_bio_mds/m3_synth_bio_mds.csv',
    'm4': './../data/md_generations/m4_synth_bio_mds/m4_synth_bio_mds.csv',
}

nodes_counter = {
    'm2': count_nodes_per_run_m2,
    'm3': count_nodes_per_run_m34,
    'm4': count_nodes_per_run_m34,
}

overall_stats = []

for p_name, p_path in md_tables.items():
    df = pd.read_csv(p_path)

    nodes = []

    for q_id in range(QUERIES_CNT):
        nodes += nodes_counter[p_name](df, q_id, RUNS)
    
    overall_stats.append({
        'exp_name': p_name, 
        'fields_cnt_avg': np.array(nodes).mean(), 
        'fields_cnt_std': np.array(nodes).std(), 
    })

stats_df = pd.DataFrame(overall_stats)
stats_df.head(10)

,exp_name,fields_cnt_avg,fields_cnt_std
0,m2,22.671875,2.713132
1,m3,12.625000,2.281036
2,m4,31.671875,9.292226


## Наконец метрики M3 vs M4 - число ошибок

In [12]:
md_folders = {
    'm3': './../data/md_generations/m3_synth_bio_mds/',
    'm4': './../data/md_generations/m4_synth_bio_mds/',
}

overall_errors = []

for p_name, p_path in md_folders.items():
    dags_errors = []
    mds_errors = []
    dag_iters = []
    md_iters = []

    for q_id in range(QUERIES_CNT):
        for run in range(RUNS):
            # count dag errors
            dag_errors = {}
            qr_errs = []
            with open(os.path.join(p_path, f'q{q_id}_r{run}', 'dag_stats.json'), 'rt') as fo:
                dag_stats = json.load(fo)
                dag_errors = dag_stats['all_errors']
            for step, errors in dag_errors.items():
                for err in errors:
                    if '400' not in err and '429' not in err:
                        qr_errs.append(err)
            dags_errors.append(len(qr_errs))
            dag_iters.append(dag_stats['reworks'] + 1)

            # count md errors
            md_errors = {}
            qr_errs = []
            with open(os.path.join(p_path, f'q{q_id}_r{run}', 'md_stats.json'), 'rt') as fo:
                md_stats = json.load(fo)
                md_errors = md_stats['all_errors']
            for err in md_errors:
                if '400' not in err and '429' not in err:
                    qr_errs.append(err)
            mds_errors.append(len(qr_errs))
            md_iters.append(md_stats['reworks'] + 1)
    
    overall_errors.append({
        'exp_name': p_name, 
        'dag_errs_avg': np.array(dags_errors).mean(), 
        'dag_errs_std': np.array(dags_errors).std(),  
        'md_errs_avg': np.array(mds_errors).mean(), 
        'md_errs_std': np.array(mds_errors).std(), 
        'dag_iters_avg': np.array(dag_iters).mean(), 
        'dag_iters_std': np.array(dag_iters).std(),  
        'md_iters_avg': np.array(md_iters).mean(), 
        'md_iters_std': np.array(md_iters).std(), 
    })

stats_df = pd.DataFrame(overall_errors)
stats_df.head(10)

,exp_name,dag_errs_avg,dag_errs_std,md_errs_avg,md_errs_std,dag_iters_avg,dag_iters_std,md_iters_avg,md_iters_std
0,m3,0.046875,0.528254,11.218750,9.939613,1.171875,0.587758,6.265625,3.615255
1,m4,3.023438,4.677848,23.171875,27.599850,2.117188,1.428620,9.382812,7.976647
